# Will this Patient be Readmitted Within 30 Days?

**Input:**
- Discharge Summary
- Prior Readmissions
- Diagnosis

**Output:** Yes/No

## Possible Models:
- Logistic Regression
- Random Forest
- Gradient Boosted Trees

In [1]:
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from pyspark.sql.functions import col, to_date, datediff, unix_timestamp, lead, when, year
from pyspark.sql.types import IntegerType
from pyspark.sql import Window
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, precision_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import requests
import json
from urllib.parse import quote
import logging
from datetime import datetime
import matplotlib.pyplot as plt

# Get the Dataset

In [2]:


# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[
        logging.FileHandler(f'synthea_load_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'),
        logging.StreamHandler()
    ]
)

def load_synthea_data(config_path=None, config_url=None, error_log_path="error_log.txt"):
    # Load config from local file or URL
    if config_url:
        config_response = requests.get(config_url)
        try:
            config = config_response.json()
        except json.JSONDecodeError:
            print("Error: Response is not valid JSON. Here is the response text:")
            print(config_response.text)
            raise
    elif config_path:
        with open(config_path, 'r') as f:
            config = json.load(f)
    else:
        raise ValueError("Must provide either config_path or config_url")
    
    base_url = config['base_url']
    files = config['files']
    
    data = {}
    for file in files:
        try:
            encoded_file = quote(file)
            url = f"{base_url}/{encoded_file}"
            df = pd.read_csv(url)
            key = file.replace('.csv', '')
            data[key] = df
            print(f"Loaded {file}: {len(df)} rows")
        except Exception as e:
            logging.error(f"Error loading {file}: {e}")

    
    return data




Loaded Synthetic ML Encounter Data.csv: 995 rows
Loaded Synthetic ML Patient Data.csv: 11 rows


In [3]:
# Load from GitHub-hosted JSON config
config_url = "https://raw.githubusercontent.com/Nicholas26-design/PortfolioProjects/main/MachineLearningProjects/Synthea/synthea_config"
dataframes = load_synthea_data(config_url=config_url)
# Load the encounter_ and patient_ dataframes from the dictionary
encounter_pandas_df = dataframes["Synthetic ML Encounter Data"]
patient_df = dataframes["Synthetic ML Patient Data"]

Loaded Synthetic ML Encounter Data.csv: 995 rows
Loaded Synthetic ML Patient Data.csv: 11 rows


# Cleaning

In [10]:
# Convert ISO 8601 string to datetime using pandas functions
encounter_df = dataframes["Synthetic ML Encounter Data"].copy()  # Create a copy to avoid modifying original

# Convert timestamp strings to datetime, then extract date
encounter_df['start_date'] = pd.to_datetime(encounter_df['start_time'])
encounter_df['end_date'] = pd.to_datetime(encounter_df['end_time'])

# Calculate duration in days
encounter_df['duration_days'] = (
    pd.to_datetime(encounter_df['end_time']) - pd.to_datetime(encounter_df['start_time'])
)

# Alternative approach keeping datetime format instead of date
# encounter_df['start_date'] = pd.to_datetime(encounter_df['start_time'])
# encounter_df['end_date'] = pd.to_datetime(encounter_df['end_time'])
# encounter_df['duration_days'] = (encounter_df['end_date'] - encounter_df['start_date']).dt.days

# Clean patient data - remove digits from names
patient_df = dataframes["Synthetic ML Patient Data"].copy()  # Create a copy
patient_df["first_name"] = patient_df["first_name"].str.replace(r"\d+", "", regex=True)
patient_df["last_name"] = patient_df["last_name"].str.replace(r"\d+", "", regex=True)
# Convert birthdate to datetime
patient_df['birth_date'] = pd.to_datetime(patient_df['birth_date'], errors='coerce')

# Ensure patient_id is string in both DataFrames
encounter_df['patient_id'] = encounter_df['patient_id'].astype(str)
patient_df['patient_id'] = patient_df['patient_id'].astype(str)

# Perform the left outer join using pandas merge
patient_encounters_df = encounter_df.merge(
    patient_df, on="patient_id", how="left"
)

# Display the result
print(patient_encounters_df)

                             encounter_id    status class_code  \
0    a354b693-ac07-2f65-690d-8bb0c1cf6ab6  finished        AMB   
1    7e8b5715-ee51-9cb0-f0eb-f5a2cd850d9f  finished        AMB   
2    3ce9d2a8-cf3c-3eb1-f81b-7efd61921964  finished        AMB   
3    1081fcfc-5d01-0db4-a4e3-517f4f6a5be2  finished       EMER   
4    28c198c4-5e1a-0213-d8bb-da116a6d4275  finished        AMB   
..                                    ...       ...        ...   
990  c0e5c026-bc74-630c-3d27-eb7f43489677  finished        AMB   
991  9b07f13e-c53f-bd2a-fa8e-6c51972006bb  finished        AMB   
992  a02a64c9-0635-da28-ec97-ed838e0fa007  finished        AMB   
993  51111149-5d38-a011-d211-8040e78ce01c  finished        AMB   
994  1bfb0b6d-d2d4-a888-8943-f40df03ae7fa  finished        AMB   

                                         class_system  \
0    http://terminology.hl7.org/CodeSystem/v3-ActCode   
1    http://terminology.hl7.org/CodeSystem/v3-ActCode   
2    http://terminology.hl7.org/Code

C:\Users\Nicholas\AppData\Local\Temp\ipykernel_16280\3164375192.py:5: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  encounter_df['start_date'] = pd.to_datetime(encounter_df['start_time'])
C:\Users\Nicholas\AppData\Local\Temp\ipykernel_16280\3164375192.py:6: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  encounter_df['end_date'] = pd.to_datetime(encounter_df['end_time'])
C:\Users\Nicholas\AppData\Local\Temp\ipykernel_16280\3164375192.py:10: Futur

# Exploratory Analysis

In [34]:
encounter_count_df = patient_encounters_df.groupby(
    ["patient_id", "first_name", "last_name"]
).size().reset_index(name='encounter_count')
print(encounter_count_df)

                              patient_id first_name   last_name  \
0   02c9b909-e1a3-7f91-6e8a-5c55a25d45f6      Sandy    O'Reilly   
1   382b52df-9bcf-c0b9-28db-4b4f79172394     Kazuko     Hackett   
2   4f648f81-8c7a-bb0a-16f8-744c028fc9fd       Cory      Mayert   
3   562a37a6-b0e0-501e-8912-63ccba146eb0  Lawerence       O'Kon   
4   57f64a10-d647-474b-9766-5a19e02b102a       Bill  Morissette   
5   8174bbcc-4b8c-d8fa-3488-2bea2fae12a0      Juana        Dach   
6   993fb8ba-c9a8-029f-db12-85574a42b7d8     Waylon       Olson   
7   b0aa9e58-e54c-0082-0c17-82c84788d5ec     Ernest        Jast   
8   d41a3cc9-bf57-e901-19a4-d4418ffa34bf      Tania   Bechtelar   
9   d5631662-a8c1-49ee-df45-5251a41117f0       Noel       Boyle   
10  f6ae995b-a00d-749b-ef2b-1881ab8b26c6     Janeen    Luettgen   

    encounter_count  
0                30  
1                34  
2                69  
3                68  
4                44  
5                32  
6                30  
7               567

In [25]:
# Convert start_time and end_time to datetime if not already
patient_encounters_df['start_time'] = pd.to_datetime(patient_encounters_df['start_time'], errors='coerce')
patient_encounters_df['end_time'] = pd.to_datetime(patient_encounters_df['end_time'], errors='coerce')

# Sort by patient_id and start_time
df_sorted = patient_encounters_df.sort_values(['patient_id', 'start_time'])

# Add next encounter's start_time per patient
df_sorted['next_start_time'] = (
    df_sorted.groupby('patient_id')['start_time'].shift(-1)
)

# Add next encounter's end_time per patient (for completeness, not strictly needed)
df_sorted['next_end_time'] = (
    df_sorted.groupby('patient_id')['end_time'].shift(-1)
)

# Calculate days until next encounter
df_sorted['days_until_next'] = (df_sorted['next_start_time'] - df_sorted['end_time'])

# Create readmission label: 1 if within 30 days, 0 otherwise
df_sorted['readmitted_within_30_days'] = (
    (df_sorted['days_until_next'] >= pd.Timedelta(days=0)) & (df_sorted['days_until_next'] <= pd.Timedelta(days=30))
).astype(int)

# Optional: check distribution
print(df_sorted['readmitted_within_30_days'].value_counts())

readmitted_within_30_days
0    549
1    446
Name: count, dtype: int64


# Feature and Target Selection

In [28]:
# Calculate age: year(start_time) - year(birth_date) using pandas
df_clean = df_sorted.copy()
df_clean['age'] = df_clean['start_time'].dt.year - df_clean['birth_date'].dt.year

# Define feature columns again
features_to_keep = [
    "status",
    "encounter_type_code",
    "encounter_type_text",
    "duration_days",
    "gender",
    "marital_status",
    "age",
]

# Define target column
target_col = "readmitted_within_30_days"

# Select relevant columns from the DataFrame
selected_cols = features_to_keep + [target_col]
df_pandas = df_clean[selected_cols]

# Drop rows with missing values
df_pandas = df_pandas.dropna()

# Split into features (X) and target (y)
X = df_pandas[features_to_keep]
y = df_pandas[target_col]

# Preprocessing

In [36]:
# Define categorical and numeric columns
categorical_features = ["status", "encounter_type_text", "gender", "marital_status"]
numeric_features = ["encounter_type_code", "duration_days", "age"]

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="mean")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

# Now X and y are ready to be used in a model pipeline

# Modeling

In [37]:
# set the experiment id
# mlflow.set_experiment(experiment_id="336962172052827")

mlflow.autolog()

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Full modeling pipeline ---
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, solver="lbfgs")),
    ]
)

# --- Train the model ---
model.fit(X_train, y_train)

# --- Evaluate ---
# By default, models predict Class 1 if the probability is greater than or equal to 0.5.

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

2025/07/02 11:11:29 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/07/02 11:11:29 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2025/07/02 11:11:29 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.
2025/07/02 11:11:29 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '2da76bf054f64619a0905acee625633f', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


TypeError: float() argument must be a string or a number, not 'Timedelta'

# Calibration

In [0]:
# Get probabilities for the positive class (Class 1)
y_probs = model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
prec_class_0 = []
prec_class_1 = []

# Compute precision for both classes at different thresholds
for t in thresholds:
    y_pred_thresh = (y_probs >= t).astype(int)
    prec_class_0.append(precision_score(y_test, y_pred_thresh, pos_label=0))
    prec_class_1.append(precision_score(y_test, y_pred_thresh, pos_label=1))

# Plot both
plt.figure(figsize=(8, 6))
plt.plot(thresholds, prec_class_0, marker='o', label='Class 0 Precision', color='blue')
plt.plot(thresholds, prec_class_1, marker='s', label='Class 1 Precision', color='green')
plt.xlabel("Threshold for Predicting Class 1")
plt.ylabel("Precision")
plt.title("Precision vs. Threshold for Both Classes")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Second Run
0.5714285714285714: Class 0 Precision (This is the most precision you get)
0.9147286821705426: Class 1 Precision (This is the precision you get when class 0 is at its most precise)
Threshold at these points: 0.65

In [0]:
# Set the experiment id
mlflow.set_experiment(experiment_id="336962172052827")

mlflow.autolog()

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Full modeling pipeline ---
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, solver="lbfgs")),
    ]
)

# --- Train the model ---
model.fit(X_train, y_train)

# --- Evaluate with specified threshold ---
y_probs = model.predict_proba(X_test)[:, 1]
threshold = 0.65
y_pred_thresh = (y_probs >= threshold).astype(int)

print(classification_report(y_test, y_pred_thresh))